In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestRegressor

# ==========================================
# 1. 設定値（コンフィグ）の定義
# ==========================================
# データのパスや列名などを一箇所にまとめます
TRAIN_PATH = '../utokyo-mlapp-2026-ex1/train.csv'
TARGET_COL = '取引価格（総額）'

# ベースラインとして、まずは扱いやすい（またはとりあえずカテゴリとして処理できる）列を選択します
FEATURE_COLS = ['市区町村名', '間取り', '面積（㎡）', '建築年', '建物の構造']

# ==========================================
# 2. データの読み込み
# ==========================================
print("データを読み込んでいます...")
train_df = pd.read_csv(TRAIN_PATH)

X = train_df[FEATURE_COLS].astype(str) 
y = train_df[TARGET_COL]

# ==========================================
# 3. 前処理パイプラインの構築
# ==========================================
# 今回は簡単のため、指定した列をすべて「カテゴリ変数」として扱い、数値に変換（Ordinal Encoding）します。
# 未知の値（テストデータのみに存在する値など）は「-1」に変換してエラーを防ぎます。
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # 欠損値を最頻値で埋める
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', categorical_transformer, FEATURE_COLS)
    ])

# ==========================================
# 4. モデルの構築（前処理 + アルゴリズム）
# ==========================================
# テストデータにも全く同じ処理を適用しやすくするため、Pipelineにまとめます
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=10, random_state=42)) # 最初は計算を早くするため決定木の数を10本に絞ります
])

# ==========================================
# 5. 交差検証（Cross Validation）とRMSLEの算出
# ==========================================
print("交差検証を開始します...")
kf = KFold(n_splits=3, shuffle=True, random_state=42) # 3分割で検証
rmsle_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # 目的変数 y の対数をとった log(y + 1) で学習させます（RMSLEを計算しやすくするため）
    y_train_log = np.log1p(y_train)
    y_val_log = np.log1p(y_val)
    
    # 学習
    model.fit(X_train, y_train_log)
    
    # 検証用データで予測
    y_pred_log = model.predict(X_val)
    
    # 対数空間でのRMSEを計算（＝元の価格空間でのRMSLEになります）
    # np.sqrt(np.mean((y_val_log - y_pred_log) ** 2)) と同等の計算です
    fold_rmsle = np.sqrt(np.mean((y_val_log - y_pred_log) ** 2))
    rmsle_scores.append(fold_rmsle)
    
    print(f"Fold {fold+1} RMSLE: {fold_rmsle:.4f}")

print(f"\nAverage RMSLE: {np.mean(rmsle_scores):.4f}")


データを読み込んでいます...


/var/folders/_5/91gsm5cs03s2zzx2b4b3795c0000gn/T/ipykernel_32703/4204484692.py:24: DtypeWarning: Columns (9: 面積（㎡）) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(TRAIN_PATH)


交差検証を開始します...
Fold 1 RMSLE: 0.2884
Fold 2 RMSLE: 0.2883
Fold 3 RMSLE: 0.2857

Average RMSLE: 0.2874
